In [5]:
import pandas as pd
import requests
from pathlib import Path
import time
import yfinance as yf

# -------------------
# Config / Paths
# -------------------
input_path = Path(r"D:\LinhDao\Programming\SUPERFUNdProject\final_data\cbus_final.csv")
openfigi_api_key = "445ba62b-3be2-4f04-8f4f-4c9ec264d72b"  # your key

# -------------------
# Load & keep only rows with Stock ID values
# -------------------
df = pd.read_csv(input_path, keep_default_na=False)  # "NA" stays literal, not NaN

if "Stock ID" not in df.columns:
    raise ValueError("File must contain a 'Stock ID' column.")

mask_has_value = df["Stock ID"].astype(str).str.strip().ne("")
df = df.loc[mask_has_value].copy()

# -------------------
# Classifier (unchanged)
# -------------------
def classify_stockid(x: str) -> str:
    if not x or str(x).strip() == "":
        return "EMPTY"
    val = str(x).strip()
    if (len(val) == 12) and val[:2].isalpha() and val[:2].isupper() and (" " not in val):
        return "ISIN"
    if " " not in val:
        if len(val) == 7:
            return "SEDOL"
        if len(val) == 6 and val.isdigit():
            return "SEDOL"
    return "BB Ticker"

df["ID_Type"] = df["Stock ID"].apply(classify_stockid)

# -------------------
# ISIN -> Yahoo Ticker via yfinance.Ticker(ISIN)
# -------------------
def yahoo_symbol_from_isin(isin: str) -> str | None:
    try:
        t = yf.Ticker(isin)
        info = t.get_info()
        if isinstance(info, dict):
            sym = info.get("symbol")
            if sym and str(sym).strip():
                return str(sym).strip()
        # optional tiny fallback
        fast = getattr(t, "fast_info", None)
        if isinstance(fast, dict):
            sym = fast.get("symbol")
            if sym and str(sym).strip():
                return str(sym).strip()
        return None
    except Exception:
        return None

isin_mask = df["ID_Type"].eq("ISIN")
isin_list = df.loc[isin_mask, "Stock ID"].astype(str).unique().tolist()

isin_to_yahoo = {}
for isin in isin_list:
    isin_to_yahoo[isin] = yahoo_symbol_from_isin(isin)
    time.sleep(0.05)

df["Yahoo Ticker"] = ""
df.loc[isin_mask, "Yahoo Ticker"] = df.loc[isin_mask, "Stock ID"].map(isin_to_yahoo).fillna("")

# -------------------
# SEDOL -> OpenFIGI (exchCode + ticker)
#   IMPORTANT: we pad to 7 chars ONLY for the API request.
#   We NEVER write the padded value back to df["Stock ID"].
# -------------------
OPENFIGI_URL = "https://api.openfigi.com/v3/mapping"

def normalize_sedol_for_lookup(original: str) -> str:
    """Return a runtime-only normalized SEDOL for API (adds leading 0 if 6 digits)."""
    s = str(original).strip()
    if s.isdigit() and len(s) == 6:
        return s.zfill(7)   # pad to 7 for the API
    return s  # 7-char SEDOLs (may be alnum) sent as-is

def openfigi_map_sedols(original_sedols: list[str]) -> dict[str, dict]:
    """
    Returns mapping keyed by the ORIGINAL SEDOL from your CSV:
      {original_sedol: {'ticker': <str or ''>, 'exchCode': <str or ''>}}
    We only pad for the API call; the original keys are preserved.
    """
    headers = {"Content-Type": "application/json"}
    if openfigi_api_key:
        headers["X-OPENFIGI-APIKEY"] = openfigi_api_key

    if not original_sedols:
        return {}

    # Build runtime-only normalized map (original -> normalized)
    norm_map = {orig: normalize_sedol_for_lookup(orig) for orig in original_sedols}

    # Reverse map normalized -> list of originals (just in case of duplicates)
    rev_map = {}
    for orig, norm in norm_map.items():
        rev_map.setdefault(norm, []).append(orig)

    # Prepare jobs using ID_SEDOL (OpenFIGI expects the ID_ prefix)
    jobs = [{"idType": "ID_SEDOL", "idValue": norm} for norm in rev_map.keys()]

    results = {orig: {"ticker": "", "exchCode": ""} for orig in original_sedols}

    # Batch in chunks of 100
    for i in range(0, len(jobs), 100):
        batch = jobs[i:i+100]
        try:
            r = requests.post(OPENFIGI_URL, json=batch, headers=headers, timeout=30)
            r.raise_for_status()
            data = r.json()
            # Align each response with its normalized idValue
            for job, resp in zip(batch, data):
                norm = job["idValue"]
                # all original SEDOLs that map to this normalized one
                originals = rev_map.get(norm, [])
                ticker = exch = ""
                if isinstance(resp, dict) and isinstance(resp.get("data"), list) and len(resp["data"]) > 0:
                    # prefer a row with both fields
                    row = next((d for d in resp["data"] if d.get("ticker") and d.get("exchCode")), resp["data"][0])
                    ticker = str(row.get("ticker") or "").strip()
                    exch   = str(row.get("exchCode") or "").strip()
                # write back under ORIGINAL keys only
                for orig in originals:
                    results[orig] = {"ticker": ticker, "exchCode": exch}
        except requests.RequestException:
            # On error, leave defaults ("")
            pass
        time.sleep(0.15)

    return results

sedol_mask = df["ID_Type"].eq("SEDOL")
sedol_originals = df.loc[sedol_mask, "Stock ID"].astype(str).unique().tolist()
sedol_map = openfigi_map_sedols(sedol_originals) if sedol_originals else {}

# Populate new columns WITHOUT touching df["Stock ID"]
df["Exchange Code"] = ""
df["Ticker"] = ""
df.loc[sedol_mask, "Exchange Code"] = df.loc[sedol_mask, "Stock ID"].map(lambda x: sedol_map.get(str(x), {}).get("exchCode", "")).fillna("")
df.loc[sedol_mask, "Ticker"]        = df.loc[sedol_mask, "Stock ID"].map(lambda x: sedol_map.get(str(x), {}).get("ticker", "")).fillna("")

# -------------------
# Summary
# -------------------
total_nonempty = len(df)
isin_found = int(df.loc[isin_mask, "Yahoo Ticker"].astype(str).str.strip().ne("").sum())
sedol_found = int((
    df.loc[sedol_mask, ["Exchange Code","Ticker"]]
      .apply(lambda r: str(r["Exchange Code"]).strip() != "" and str(r["Ticker"]).strip() != "", axis=1)
).sum())

success_total = isin_found + sedol_found
success_rate = (success_total / total_nonempty * 100) if total_nonempty else 0.0

print(f"✅ Processed file: {input_path}")
print(f"Rows with non-empty Stock ID: {total_nonempty}")
print(f"  ISIN rows:  {int(isin_mask.sum())}  | Yahoo tickers found: {isin_found}")
print(f"  SEDOL rows: {int(sedol_mask.sum())} | exchCode+ticker found: {sedol_found}")
print(f"  ➤ Overall success: {success_total} / {total_nonempty} = {success_rate:.2f}%")

# -------------------
# Save trimmed output (original Stock ID values preserved)
# -------------------
cols_to_keep = [
    "Option Name",
    "Asset Class Name",
    "Name/Kind of Investment Item",
    "Stock ID",          # <-- ORIGINAL, unchanged
    "ID_Type",
    "Listed Country",
    "Exchange Code",
    "Ticker",
    "Yahoo Ticker",
]
existing = [c for c in cols_to_keep if c in df.columns]
output_path = input_path.with_name(input_path.stem + "_enriched.csv")
df[existing].to_csv(output_path, index=False, na_rep="")
print(f"📝 Enriched file saved as: {output_path}")


✅ Processed file: D:\LinhDao\Programming\SUPERFUNdProject\final_data\cbus_final.csv
Rows with non-empty Stock ID: 1964
  ISIN rows:  5  | Yahoo tickers found: 4
  SEDOL rows: 1959 | exchCode+ticker found: 1932
  ➤ Overall success: 1936 / 1964 = 98.57%
📝 Enriched file saved as: D:\LinhDao\Programming\SUPERFUNdProject\final_data\cbus_final_enriched.csv


In [7]:
import pandas as pd
from pathlib import Path
import re

# === Paths ===
enriched_path = Path(r"D:\LinhDao\Programming\SUPERFUNdProject\final_data\cbus_final_enriched.csv")
bbg_path      = Path(r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\bloomberg-exchange-codes-full.csv")
yahoo_path    = Path(r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\yahoo_suffix_mapping_full.csv")

# === Load enriched file (keep "NA" literal) ===
df = pd.read_csv(enriched_path, keep_default_na=False)

for col in ("Exchange Code", "Yahoo Ticker", "Listed Country"):
    if col not in df.columns:
        raise ValueError(f"Missing column '{col}' in enriched file. Got: {list(df.columns)}")

# === Load lookup tables (BOM-safe headers) ===
bbg   = pd.read_csv(bbg_path,   keep_default_na=False, encoding="utf-8-sig")
yahoo = pd.read_csv(yahoo_path, keep_default_na=False, encoding="utf-8-sig")

for col in ("BBG_Code", "Composite_Code", "Country (Friendly)"):
    if col not in bbg.columns:
        raise ValueError(f"Bloomberg file missing '{col}'. Headers: {list(bbg.columns)}")
for col in ("Suffix", "Country"):
    if col not in yahoo.columns:
        raise ValueError(f"Yahoo suffix file missing '{col}'. Headers: {list(yahoo.columns)}")

# === Helper: take the FIRST TWO CHARACTERS (letters or digits), case-insensitive ===
def first2(s: str) -> str:
    return str(s).strip().upper()[:2]

# --- Build Bloomberg map: (first2 of BBG_Code / Composite_Code) -> Country (Friendly)
bbg_map = {}
for _, r in bbg.iterrows():
    country = str(r["Country (Friendly)"]).strip()
    # prefer BBG_Code; only use Composite_Code if that key not set yet
    c_bbg = first2(r["BBG_Code"])
    if c_bbg and country and c_bbg not in bbg_map:
        bbg_map[c_bbg] = country
    c_comp = first2(r["Composite_Code"])
    if c_comp and country and c_comp not in bbg_map:
        bbg_map[c_comp] = country

# --- Build Yahoo suffix -> Country map (empty suffix "" → U.S. row)
yahoo["Suffix"] = yahoo["Suffix"].astype(str)  # keep blanks as ""
yahoo_map = dict(zip(yahoo["Suffix"].map(lambda s: s.strip()), yahoo["Country"]))

def extract_suffix(t: str) -> str:
    t = str(t).strip()
    if "." in t:
        return t[t.rfind("."):]  # e.g., ".AX"
    return ""                    # no suffix → empty string key

# === Apply from Exchange Code (using FIRST TWO CHARACTERS) ===
before = df["Listed Country"].copy()

ex_mask = df["Exchange Code"].astype(str).str.strip().ne("")
ex_key  = df.loc[ex_mask, "Exchange Code"].map(first2)
ex_country = ex_key.map(lambda k: bbg_map.get(k, ""))

ex_hits = int(ex_country.astype(str).str.strip().ne("").sum())
df.loc[ex_mask & ex_country.astype(str).str.strip().ne(""), "Listed Country"] = ex_country

# === Apply from Yahoo Ticker suffix ===
yt_mask = df["Yahoo Ticker"].astype(str).str.strip().ne("")
yt_suffix = df.loc[yt_mask, "Yahoo Ticker"].map(extract_suffix)
yt_country = yt_suffix.map(lambda s: yahoo_map.get(s, ""))

yt_hits = int(yt_country.astype(str).str.strip().ne("").sum())
df.loc[yt_mask & yt_country.astype(str).str.strip().ne(""), "Listed Country"] = yt_country

# === Metrics & save IN PLACE ===
changed = int((df["Listed Country"] != before).sum())

print(f"Exchange Code rows: {int(ex_mask.sum())} | matched via Bloomberg (first2): {ex_hits}")
print(f"Yahoo Ticker rows:  {int(yt_mask.sum())} | matched via suffix map:       {yt_hits}")
print(f"Rows where 'Listed Country' changed: {changed}")

df.to_csv(enriched_path, index=False, na_rep="")
print(f"✅ Saved in place: {enriched_path}")


Exchange Code rows: 1932 | matched via Bloomberg (first2): 1932
Yahoo Ticker rows:  4 | matched via suffix map:       4
Rows where 'Listed Country' changed: 114
✅ Saved in place: D:\LinhDao\Programming\SUPERFUNdProject\final_data\cbus_final_enriched.csv
